# E3 y E4. Calidad y aporte de la síntesis
Ejecuta esta fase tras completar DRAEM-Ref. Congela los pares revisados antes de comparar las variantes. Mantén las semillas, el presupuesto y el TEST de referencia.

In [ ]:
from pathlib import Path
from IPython import get_ipython

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)


def load_notebook(name):
    path = ROOT / "notebooks" / "shared" / (name + ".ipynb")
    get_ipython().run_line_magic("run", f'-i "{path.as_posix()}"')


load_notebook("workspace")

In [ ]:
load_notebook("experiment")
load_notebook("metrics")
load_notebook("synthetic")
frame, dataset_hash = cap_rows()
mapping, pair_hash, coverage = synthetic_pairs(frame)
display(coverage)
print("Normales de TRAIN con pares sintéticos:", len(mapping))
print("Huella del conjunto sintético:", pair_hash)

In [ ]:
variant = "draem-project"
seed = 17
config = json.loads((ROOT / "configs/models/draem-ref.json").read_text())
if variant not in ("draem-project", "draem-mixed"):
    raise ValueError("Variante desconocida.")
run = fit_run(config, "synthetic", seed, variant=variant)

## Comparación pareada
Después de registrar las tres semillas de cada variante elegida, selecciona sus identificadores antes de abrir TEST. El conjunto de referencia se toma del plan final congelado.

In [ ]:
synthetic_runs = []
if synthetic_runs:
    suite = RUNS.parent / "final" / dataset_hash
    original = json.loads((suite / "plan.json").read_text())["runs"]
    reference = {}
    for name in original:
        record = json.loads((RUNS / name / "record.json").read_text(encoding="utf-8"))
        if record["config"] == config:
            reference[record["seed"]] = json.loads(
                (suite / name / "metrics.json").read_text()
            )
    selected = {}
    for name in synthetic_runs:
        if not re.fullmatch(r"[0-9TZ]+-[a-f0-9]{8}", name):
            raise ValueError("Identificador de corrida inválido.")
        run = RUNS / name
        rec = json.loads((run / "record.json").read_text(encoding="utf-8"))
        if (
            rec["status"] != "PASS"
            or rec["stage"] != "synthetic"
            or rec["dataset_hash"] != dataset_hash
            or rec["implementation_hash"] != implementation_hash()
            or rec["pair_hash"] != pair_hash
            or rec["config"] != config
            or rec["seed"] not in reference
            or rec["lock_hash"] != digest(ROOT / "uv.lock")
        ):
            raise ValueError("La corrida no es comparable con la referencia.")
        selected[name] = (rec["variant"], rec["seed"])
    for group in set(v for v, s in selected.values()):
        seeds = [s for v, s in selected.values() if v == group]
        if sorted(seeds) != sorted(config["seeds"]):
            raise ValueError("Cada variante requiere las tres semillas.")
    study = suite / "synthetic" / pair_hash
    study.mkdir(parents=True, exist_ok=True)
    freeze = {
        name: {
            f: digest(RUNS / name / f)
            for f in ("record.json", "threshold.json", "weights.pt")
        }
        for name in selected
    }
    plan_path = study / "plan.json"
    if plan_path.exists() and json.loads(plan_path.read_text()) != freeze:
        raise ValueError("La selección del estudio sintético ya fue congelada.")
    if not plan_path.exists():
        save_json(plan_path, freeze)
    export_study_file(study, "plan.json")
    differences = []
    for name, (variant, seed) in selected.items():
        run, folder = RUNS / name, study / name
        if not (folder / "metrics.json").exists():
            model, transform = build_model(config)
            model.load_state_dict(
                torch.load(run / "weights.pt", map_location="cpu", weights_only=True)
            )
            threshold = json.loads((run / "threshold.json").read_text())["value"]
            torch.cuda.reset_peak_memory_stats()
            assess(
                model,
                transform,
                frame[frame.split == "test"],
                config,
                folder,
                threshold,
            )
            del model
            torch.cuda.empty_cache()
        export_study_file(folder, "metrics.json")
        metrics = json.loads((folder / "metrics.json").read_text())
        for metric in (
            "image_auroc",
            "image_ap",
            "pixel_auroc",
            "pixel_ap",
            "au_pro",
            "f1",
        ):
            differences.append(
                dict(
                    variant=variant,
                    seed=seed,
                    metric=metric,
                    delta=metrics[metric] - reference[seed][metric],
                )
            )
    pd.DataFrame(differences).to_csv(study / "paired-deltas.csv", index=False)
    export_study_file(study, "paired-deltas.csv")
    display(pd.DataFrame(differences))